In [3]:
import netCDF4 as nc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import os
import xarray as xr
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import pyproj
from scipy.interpolate import griddata
from matplotlib.axes import Axes
from cartopy.mpl.geoaxes import GeoAxes
GeoAxes._pcolormesh_patched = Axes.pcolormesh
from scipy.interpolate import griddata
import cartopy.util as cutil
from scipy.spatial import cKDTree


In [9]:
home_path = os.path.expanduser("~")

path = '/Library/Application Support/MathWorks/MATLAB/Add-Ons/Toolboxes/DASH/@optimalSensor'
path2 = '/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - MacBook Air/Python/Ice Cores/data/model/ccsm4_last_millennium'
path3 = '/DataFiles'


In [11]:
ds = nc.Dataset(home_path + path3 +"/pr_sfc_Amon_CCSM4_past1000_085001-185012.nc")
tas = nc.Dataset(home_path + path3 +"/tas_sfc_Amon_CCSM4_past1000_085001-185012.nc")
lat = np.array(ds.variables['lat'])
lon = np.array(ds.variables['lon'])
tempPrecip = np.array(ds.variables['pr'])
tempTemp = np.array(tas.variables['tas'])

/var/folders/4d/5th1wr_s21g614c74mcbh9kc0000gn/T/ipykernel_25482/4271342169.py:3: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  lat = np.array(ds.variables['lat'])
/var/folders/4d/5th1wr_s21g614c74mcbh9kc0000gn/T/ipykernel_25482/4271342169.py:4: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  lon = np.array(ds.variables['lon'])
/var/folders/4d/5th1wr_s21g614c74mcbh9kc0000gn/T/ipykernel_25482/4271342169.py:5: DeprecationWarning: __array__ implementation doesn't accept a copy k

In [13]:
era5 = nc.Dataset(home_path + path3 + "/data.nc")
firstMask = era5.variables['t2m']
tempMask = np.roll(firstMask, 1800, axis=2)
latInd = np.array(pd.read_csv(home_path + path3 +'/latInd.csv'))
erLat = era5.variables['latitude']
erLon = era5.variables['longitude']

In [14]:
temp = np.zeros([1001, 192, 288])
precip = np.zeros([1001, 192, 288])

for i in range(1001):
    temp[i] = np.mean(tempTemp[i*12:(i+1)*12], axis=0)
    precip[i] = np.mean(tempPrecip[i*12:(i+1)*12], axis=0) * 31536000





tempRegrid = np.zeros([32, 3600])
secondRegrid = np.zeros([32, 7200])


i = 0
for item in latInd:
    tempRegrid[i, :] = tempMask[0, item[0]]
    i = i + 1


for i in range(32):
    secondRegrid[i, :7198] = np.dstack((tempRegrid[i, :-1], tempRegrid[i, :-1] + np.diff(tempRegrid[i]) / 2.0)).ravel()




regrid = secondRegrid[:, ::25]



latLand = np.zeros([5794])
lonLand = np.zeros([5794])
prLand = np.zeros([5794, 1001])
tasLand = np.zeros([5794, 1001])

gridCount = 0

for i in range(32):
    for j in range(288):
        if regrid[i, j] > 0:
            latLand[gridCount] = lat[i]
            lonLand[gridCount] = lon[j]
            prLand[gridCount] = precip[:, i, j]
            tasLand[gridCount] = temp[:, i, j]
            gridCount = gridCount + 1

